# Pavement Distress Feature Extraction, sPCI Calibration & Validation Analysis
**Step 4B: Full Features, ASTM D6433 Rating Bands, 7-Group Sensitivity, Distributions & Noise Floor Sweep**
- **Model**: ResNet-34 U-Net (Final Frozen Weights from Naive Run B, Seed 42)
- **Execution**: Kaggle GPU (T4/P100), Flip-TTA
- **Artifacts Generated**:
  1. `manifests/features_gt_trainval.csv` (pruned & unpruned morphological features across 1,486 train+val GT masks)
  2. `val_predictions.npz` (compressed validation predicted & GT masks for reproducible downstream evaluation)
  3. `step_4b_summary.json` (complete structured metrics payload)

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================
import os
import sys
import subprocess
import shutil
import time
import json
from pathlib import Path

print("Installing required dependencies (segmentation-models-pytorch, albumentations, scipy, pyyaml)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0", "albumentations==2.0.8", "scipy==1.17.1", "pyyaml==6.0.2"], check=True)

# Clone repository
repo_dir = Path("/kaggle/working/repo")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

print("Cloning repository from GitHub...")
subprocess.run(["git", "clone", "https://github.com/skilloutofbounds/road-pci-intelligence.git", str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

import torch
import torchvision
import numpy as np
import pandas as pd
import cv2
import yaml
from PIL import Image
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import segmentation_models_pytorch as smp

print("=" * 80)
print("ENVIRONMENT INFORMATION:")
print(f"  • Python:       {sys.version.split()[0]}")
print(f"  • PyTorch:      {torch.__version__}")
print(f"  • Torchvision:  {torchvision.__version__}")
print(f"  • SMP Version:  {smp.__version__}")
print(f"  • CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  • GPU Device:   {torch.cuda.get_device_name(0)}")
print("=" * 80)

In [ ]:
# ==============================================================================
# 2. RESOLVE DATASET ROOTS & VERIFY FINAL WEIGHTS
# ==============================================================================
import hashlib
from src.build_manifest import DATASET_SPECS, resolve_dataset_root
from src.pavement_features import compute_features, rescale_mask, compute_skeleton_length, prune_skeleton
from src.data_rules import load_standardized_mask
from src.datasets import RoadDistressDataset
from src.spci import compute_spci, load_spci_config, get_condition_class

dataset_roots = {}
for src in ['crack500', 'road_crack', 'pothole']:
    meta = DATASET_SPECS[src]
    resolved = resolve_dataset_root(meta['owner'], meta['name'])
    if resolved is None:
        raise FileNotFoundError(f"CRITICAL: Failed to locate dataset root for {src}")
    dataset_roots[src] = resolved
    print(f"  • {src:<12} -> {resolved}")

weights_candidates = [
    Path("/kaggle/input/datasets/vedshah04/road-pci-seg-final-weights/best.pt"),
    Path("/kaggle/input/road-pci-seg-final-weights/best.pt"),
    repo_dir / "scratch/downloaded_final_weights/best.pt"
]
weights_path = None
for p in weights_candidates:
    if p.exists():
        weights_path = p
        break

if weights_path is None:
    raise FileNotFoundError("CRITICAL: Failed to locate frozen weights best.pt in input datasets.")

hasher = hashlib.sha256()
with open(weights_path, 'rb') as f:
    while chunk := f.read(65536):
        hasher.update(chunk)
sha256 = hasher.hexdigest()

EXPECTED_SHA256 = "9269c5edb6e5d37b359bd3337fbd83c60901cd2b241f9ef789e9116fee400ef7"
print(f"[OK] Weights located at: {weights_path}")
print(f"  • SHA256: {sha256}")
assert sha256 == EXPECTED_SHA256, f"Weight SHA256 mismatch! Expected {EXPECTED_SHA256}, got {sha256}"

In [ ]:
# ==============================================================================
# 3. PHASE 1: GROUND-TRUTH FEATURE EXTRACTION (TRAIN + VAL)
# ==============================================================================
manifest_path = repo_dir / "manifests/segmentation_manifest.csv"
manifest_df = pd.read_csv(manifest_path)
trainval_df = manifest_df[manifest_df['split'].isin(['train', 'val'])].reset_index(drop=True)
print(f"Extracting features for {len(trainval_df)} train+val GT masks...")

gt_feature_records = []
all_crack_local_widths = []

t0 = time.time()
for idx, row in trainval_df.iterrows():
    src = row['source']
    split = row['split']
    root = dataset_roots[src]
    mask_full = root / row['mask_path_or_label_path']
    img_shape = (int(row['height']), int(row['width']))
    
    raw_mask = load_standardized_mask(src, mask_full, img_shape=img_shape)
    
    if src in ['crack500', 'road_crack']:
        crack_m = raw_mask
        pothole_m = None
    elif src == 'pothole':
        crack_m = None
        pothole_m = raw_mask
    else:
        crack_m = None
        pothole_m = None
        
    # Compute both raw (unpruned) and pruned features
    feats_raw = compute_features(crack_mask=crack_m, pothole_mask=pothole_m, prune_branches=False)
    feats_pruned = compute_features(crack_mask=crack_m, pothole_mask=pothole_m, prune_branches=True, min_branch_len=12)
    
    # Collect local widths for crack-bearing masks
    if src in ['crack500', 'road_crack'] and feats_pruned['crack_length_px'] > 0:
        c_res = rescale_mask(crack_m, 512)
        n_lbl, lbls, stats, _ = cv2.connectedComponentsWithStats(c_res, connectivity=8)
        clean_c = np.zeros_like(c_res)
        for l in range(1, n_lbl):
            if stats[l, cv2.CC_STAT_AREA] >= 30:
                clean_c[lbls == l] = 1
        if np.any(clean_c):
            dt = cv2.distanceTransform(clean_c, cv2.DIST_L2, 5)
            from skimage.morphology import skeletonize
            sk = prune_skeleton(skeletonize(clean_c > 0), min_branch_len=12)
            if np.any(sk):
                widths = 2.0 * dt[sk > 0]
                all_crack_local_widths.extend(widths.tolist())
    
    rec = {
        'source': src,
        'image_path': row['image_path'],
        'split': split,
        'group_id': row['group_id'],
        # Pruned metrics (primary)
        **feats_pruned,
        # Before-pruning metrics for comparison
        'raw_n_junctions': feats_raw['n_junctions'],
        'raw_network_index': feats_raw['network_index'],
    }
    gt_feature_records.append(rec)
    
    if (idx + 1) % 400 == 0 or (idx + 1) == len(trainval_df):
        print(f"  Processed {idx + 1:,} / {len(trainval_df):,} masks in {time.time()-t0:.1f}s")

gt_features_df = pd.DataFrame(gt_feature_records)
print(f"[OK] Extracted features for {len(gt_features_df):,} masks in {time.time()-t0:.2f}s")

# Save features CSV
out_csv = repo_dir / "manifests/features_gt_trainval.csv"
gt_features_df.to_csv(out_csv, index=False)
gt_features_df.to_csv(Path("/kaggle/working/features_gt_trainval.csv"), index=False)
print(f"[OK] Saved features to {out_csv}")

In [ ]:
# ==============================================================================
# 4. DISTRIBUTION TABLES & BEFORE VS AFTER PRUNING COMPARISON
# ==============================================================================
print("=" * 105)
print("BEFORE VS. AFTER SKELETON PRUNING (< 12 PX SPURS REMOVED): JUNCTIONS & NETWORK INDEX")
print("=" * 105)
print(f"{'Source':<12} | {'Metric':<20} | {'Raw Median':<14} | {'Pruned Median':<14} | {'Raw P90':<12} | {'Pruned P90':<12}")
print("-" * 105)

for src in ['crack500', 'road_crack', 'pothole']:
    src_df = gt_features_df[gt_features_df['source'] == src]
    
    raw_j_med = np.median(src_df['raw_n_junctions'])
    pruned_j_med = np.median(src_df['n_junctions'])
    raw_j_p90 = np.percentile(src_df['raw_n_junctions'], 90)
    pruned_j_p90 = np.percentile(src_df['n_junctions'], 90)
    
    raw_ni_med = np.median(src_df['raw_network_index'])
    pruned_ni_med = np.median(src_df['network_index'])
    raw_ni_p90 = np.percentile(src_df['raw_network_index'], 90)
    pruned_ni_p90 = np.percentile(src_df['network_index'], 90)
    
    print(f"{src:<12} | {'n_junctions':<20} | {raw_j_med:14.1f} | {pruned_j_med:14.1f} | {raw_j_p90:12.1f} | {pruned_j_p90:12.1f}")
    print(f"{src:<12} | {'network_index':<20} | {raw_ni_med:14.3f} | {pruned_ni_med:14.3f} | {raw_ni_p90:12.3f} | {pruned_ni_p90:12.3f}")
    print("-" * 105)

# Empirical Crack Severity Thresholds
crack_widths_arr = np.array(all_crack_local_widths)
p33_width = float(np.percentile(crack_widths_arr, 33))
p80_width = float(np.percentile(crack_widths_arr, 80))
median_width = float(np.median(crack_widths_arr))

# Empirical n_ref
crack_bearing_df = gt_features_df[(gt_features_df['source'].isin(['crack500', 'road_crack'])) & (gt_features_df['crack_length_px'] > 0)]
n_ref_val = float(np.percentile(crack_bearing_df['network_index'], 90))

print("\n" + "=" * 80)
print("EMPIRICAL CALIBRATION THRESHOLDS (CRACK-BEARING TRAIN+VAL GT MASKS):")
print(f"  • Total Skeleton Pixels Sampled: {len(crack_widths_arr):,}")
print(f"  • Local Width 33rd Percentile (T_low):   {p33_width:.3f} px")
print(f"  • Local Width Median:                    {median_width:.3f} px")
print(f"  • Local Width 80th Percentile (T_high):  {p80_width:.3f} px")
print(f"  • Network Index 90th Percentile (n_ref): {n_ref_val:.3f}")
print("=" * 80)

In [ ]:
# ==============================================================================
# 5. VISUAL INSPECTION: 6 VAL SAMPLES PER SOURCE (IMAGE | MASK | PRUNED SKELETON)
# ==============================================================================
from skimage.morphology import skeletonize

val_df = trainval_df[trainval_df['split'] == 'val'].reset_index(drop=True)

for src in ['crack500', 'road_crack', 'pothole']:
    src_val = val_df[val_df['source'] == src].reset_index(drop=True)
    n_total = len(src_val)
    step = max(1, n_total // 6)
    indices = [min(i * step, n_total - 1) for i in range(6)]
    indices = sorted(list(dict.fromkeys(indices)))
    while len(indices) < 6:
        indices.append(len(indices))
        
    samples = src_val.iloc[indices]
    
    fig, axes = plt.subplots(6, 3, figsize=(15, 24))
    plt.subplots_adjust(top=0.96, bottom=0.03, hspace=0.32, wspace=0.10)
    fig.suptitle(f"Pruned Feature Inspection: {src.upper()} Validation Set (6 Samples)", fontsize=14, fontweight='bold', y=0.98)
    
    for row_idx, (_, row) in enumerate(samples.iterrows()):
        root = dataset_roots[src]
        img_full = root / row['image_path']
        mask_full = root / row['mask_path_or_label_path']
        img_shape = (int(row['height']), int(row['width']))
        
        with Image.open(img_full) as pil_img:
            img_rgb = np.array(pil_img.convert('RGB'))
            
        raw_mask = load_standardized_mask(src, mask_full, img_shape=img_shape)
        mask_512 = rescale_mask(raw_mask, target_shorter_side=512)
        h5, w5 = mask_512.shape[:2]
        img_512 = cv2.resize(img_rgb, (w5, h5), interpolation=cv2.INTER_LINEAR)
        
        stem = Path(row['image_path']).name
        if len(stem) > 24:
            stem = stem[:21] + "..."
            
        axes[row_idx, 0].imshow(img_512)
        axes[row_idx, 0].set_title(f"[Sample {row_idx+1}] {stem}\nImage ({w5}x{h5})", fontsize=8.5, fontweight='bold')
        axes[row_idx, 0].axis('off')
        
        mask_overlay = img_512.copy()
        color = [230, 30, 30] if src != 'pothole' else [30, 110, 240]
        fg = (mask_512 > 0)
        mask_overlay[fg] = (mask_overlay[fg].astype(float) * 0.45 + np.array(color) * 0.55).astype(np.uint8)
        axes[row_idx, 1].imshow(mask_overlay)
        axes[row_idx, 1].set_title(f"GT Mask ({'crack' if src != 'pothole' else 'pothole'})\nPixels: {int(fg.sum()):,} ({fg.mean()*100:.2f}%)", fontsize=8.5)
        axes[row_idx, 1].axis('off')
        
        skel_canvas = img_512.copy()
        if src != 'pothole':
            n_lbl, lbls, stats, _ = cv2.connectedComponentsWithStats(mask_512, connectivity=8)
            clean_crack = np.zeros_like(mask_512)
            for l in range(1, n_lbl):
                if stats[l, cv2.CC_STAT_AREA] >= 30:
                    clean_crack[lbls == l] = 1
            skel_raw = skeletonize(clean_crack > 0)
            skel = prune_skeleton(skel_raw, min_branch_len=12)
            skel_canvas[skel] = [0, 255, 0]
            
            sk_u8 = skel.astype(np.uint8)
            k_n = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]], dtype=np.uint8)
            nc = cv2.filter2D(sk_u8, -1, k_n) * sk_u8
            junc_pts = ((nc >= 3) & (sk_u8 > 0))
            jy, jx = np.where(junc_pts)
            for y_j, x_j in zip(jy, jx):
                cv2.circle(skel_canvas, (x_j, y_j), 4, (255, 0, 0), thickness=-1)
                
            length_val = compute_skeleton_length(skel)
            num_j_lbl, _ = cv2.connectedComponents(junc_pts.astype(np.uint8), connectivity=8)
            n_j = max(0, num_j_lbl - 1)
            axes[row_idx, 2].imshow(skel_canvas)
            axes[row_idx, 2].set_title(f"Pruned Skeleton & Junctions\nLength: {length_val:.1f} px | Junctions: {n_j}", fontsize=8.5, fontweight='bold')
        else:
            n_lbl, lbls, stats, _ = cv2.connectedComponentsWithStats(mask_512, connectivity=8)
            p_cnt = 0
            for l in range(1, n_lbl):
                if stats[l, cv2.CC_STAT_AREA] >= 50:
                    p_cnt += 1
                    comp_mask = (lbls == l).astype(np.uint8)
                    contours, _ = cv2.findContours(comp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(skel_canvas, contours, -1, (255, 255, 0), 2)
            axes[row_idx, 2].imshow(skel_canvas)
            axes[row_idx, 2].set_title(f"Pothole Contours (Yellow)\nValid Potholes: {p_cnt}", fontsize=8.5, fontweight='bold')
        axes[row_idx, 2].axis('off')
        
    out_fig = Path(f"/kaggle/working/val_feature_skeletons_{src}.png")
    plt.savefig(out_fig, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"[OK] Saved figure: {out_fig}")

In [ ]:
# ==============================================================================
# 6. PHASE 3: FAST GPU INFERENCE, VAL PREDICTIONS NPZ & BASELINE EVALUATION
# ==============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Inference device: {device}")

model = smp.Unet(encoder_name='resnet34', classes=2)
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

val_dataset = RoadDistressDataset(manifest_df=manifest_df, dataset_roots=dataset_roots, split='val', crop_size=512)
print(f"Validation DataLoader: {len(val_dataset)} images")

t0_eval = time.time()
paired_val_records = []

# Load committed sPCI config from repo
spci_cfg = load_spci_config(repo_dir / "configs/spci_params.yaml")
print("Active sPCI Parameters from configs/spci_params.yaml:")
print(f"  • Crack Severity Thresholds: low={spci_cfg['crack_severity']['low_threshold_px']}, high={spci_cfg['crack_severity']['high_threshold_px']}")
print(f"  • Network Factor: n_ref={spci_cfg['network_factor']['n_ref']}, beta={spci_cfg['network_factor']['beta']}")
print(f"  • Deduct Curves: {list(spci_cfg['deduct_curves'].keys())}")
print(f"  • Condition Bands: {spci_cfg.get('condition_classes')}")

with torch.no_grad():
    for idx in range(len(val_dataset)):
        image_t, target_t, label_mask, metas = val_dataset[idx]
        image_b = image_t.unsqueeze(0).to(device)
        src = metas['source']
        unpad_info = metas['eval_unpad']
        y0, x0, h, w = unpad_info
        
        # Forward Flip-TTA on GPU
        probs_orig = torch.sigmoid(model(image_b))
        probs_h = torch.flip(torch.sigmoid(model(torch.flip(image_b, dims=[-1]))), dims=[-1])
        probs_v = torch.flip(torch.sigmoid(model(torch.flip(image_b, dims=[-2]))), dims=[-2])
        probs_tta = ((probs_orig + probs_h + probs_v) / 3.0).cpu()
        
        p_crack = (probs_tta[0, 0, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8)
        p_pot = (probs_tta[0, 1, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8)
        
        t_crack = (target_t[0, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8) if label_mask[0] else None
        t_pot = (target_t[1, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8) if label_mask[1] else None
        
        # Compute pruned features
        gt_feats = compute_features(t_crack, t_pot, prune_branches=True, min_branch_len=12)
        pred_feats = compute_features(p_crack, p_pot, prune_branches=True, min_branch_len=12)
        
        # Compute sPCI with baseline config
        spci_pred = compute_spci(p_crack, p_pot, config=spci_cfg)
        spci_gt = compute_spci(t_crack, t_pot, config=spci_cfg)
        
        paired_val_records.append({
            'source': src,
            'image_path': metas['image_path'],
            'gt': gt_feats,
            'pred': pred_feats,
            'spci_gt': spci_gt['spci'],
            'spci_pred': spci_pred['spci'],
            'pred_p_crack': p_crack,
            'pred_p_pot': p_pot,
            'gt_t_crack': t_crack,
            'gt_t_pot': t_pot,
        })
        
        if (idx + 1) % 50 == 0 or (idx + 1) == len(val_dataset):
            print(f"  Evaluated {idx + 1} / {len(val_dataset)} images ({((idx+1)/len(val_dataset))*100:.1f}%) in {time.time()-t0_eval:.1f}s")

print(f"[OK] Completed validation inference & feature extraction in {time.time()-t0_eval:.2f}s")

# Save compressed val_predictions.npz in /kaggle/working/ ONLY (not in git)
npz_data = {}
for i, rec in enumerate(paired_val_records):
    npz_data[f"pred_crack_{i:03d}"] = rec['pred_p_crack']
    npz_data[f"pred_pot_{i:03d}"] = rec['pred_p_pot']
    if rec['gt_t_crack'] is not None:
        npz_data[f"gt_crack_{i:03d}"] = rec['gt_t_crack']
    if rec['gt_t_pot'] is not None:
        npz_data[f"gt_pot_{i:03d}"] = rec['gt_t_pot']

npz_data['image_paths'] = np.array([r['image_path'] for r in paired_val_records])
npz_data['sources'] = np.array([r['source'] for r in paired_val_records])
npz_path = Path("/kaggle/working/val_predictions.npz")
np.savez_compressed(npz_path, **npz_data)
print(f"[OK] Saved {npz_path} ({npz_path.stat().st_size / 1024 / 1024:.2f} MB)")

# Spearman Rank Correlation Table for Features
eval_targets = {
    'crack500': ['crack_length_px', 'relative_width', 'network_index'],
    'road_crack': ['crack_length_px', 'relative_width', 'network_index'],
    'pothole': ['pothole_area_frac']
}

print("\n" + "=" * 70)
print("GROUND-TRUTH vs. PREDICTED FEATURE RANK CORRELATION (VAL SPLIT)")
print("=" * 70)
print(f"{'Source':<12} | {'N Images':<10} | {'Feature Name':<22} | {'Spearman rho':<14}")
print("-" * 70)

feat_corr_results = {}
for src, feats in eval_targets.items():
    feat_corr_results[src] = {}
    src_recs = [r for r in paired_val_records if r['source'] == src]
    n_img = len(src_recs)
    for feat in feats:
        gt_vals = np.array([r['gt'][feat] for r in src_recs])
        pred_vals = np.array([r['pred'][feat] for r in src_recs])
        if np.all(gt_vals == gt_vals[0]) or np.all(pred_vals == pred_vals[0]):
            rho = 0.0
        else:
            rho = float(spearmanr(gt_vals, pred_vals).statistic)
        feat_corr_results[src][feat] = rho
        print(f"{src:<12} | {n_img:<10} | {feat:<22} | {rho:14.4f}")
    print("-" * 70)
print("=" * 70)

# Baseline sPCI Comparison Table
print("\n" + "=" * 70)
print("sPCI VALIDATION COMPARISON: GROUND-TRUTH vs. PREDICTED MASKS")
print("=" * 70)
print(f"{'Source':<12} | {'N Images':<10} | {'Spearman rho':<14} | {'MAE (Points)':<14}")
print("-" * 70)

spci_comparison_results = {}
for src in ['crack500', 'road_crack', 'pothole']:
    src_recs = [r for r in paired_val_records if r['source'] == src]
    n_img = len(src_recs)
    gt_spci = np.array([r['spci_gt'] for r in src_recs])
    pred_spci = np.array([r['spci_pred'] for r in src_recs])
    
    rho = float(spearmanr(gt_spci, pred_spci).statistic)
    mae = float(np.mean(np.abs(gt_spci - pred_spci)))
    spci_comparison_results[src] = {'n': n_img, 'rho': rho, 'mae': mae}
    print(f"{src:<12} | {n_img:<10} | {rho:14.4f} | {mae:14.2f}")
print("=" * 70)

In [ ]:
# ==============================================================================
# 7. SENSITIVITY ANALYSIS: 7 SEPARATED PARAMETER GROUPS (50 DRAWS EACH)
# ==============================================================================
import copy

print("\n" + "=" * 80)
print("SENSITIVITY ANALYSIS: 7 GROUPS SEPARATED (50 DRAWS EACH, PERTURBATION IN [0.7, 1.3])")
print("=" * 80)

rng = np.random.RandomState(42)
val_preds = [(r['pred_p_crack'], r['pred_p_pot']) for r in paired_val_records]
baseline_spci_vals = np.array([r['spci_pred'] for r in paired_val_records])

groups = [
    'crack a',
    'crack b',
    'pothole a',
    'pothole b',
    'severity thresholds',
    'beta',
    'network n_ref'
]

group_results = {}

for grp in groups:
    rhos = []
    for draw in range(50):
        cfg_p = copy.deepcopy(spci_cfg)
        
        if grp == 'crack a':
            for k in ['crack_low', 'crack_medium', 'crack_high']:
                cfg_p['deduct_curves'][k]['a'] *= rng.uniform(0.7, 1.3)
        elif grp == 'crack b':
            for k in ['crack_low', 'crack_medium', 'crack_high']:
                cfg_p['deduct_curves'][k]['b'] *= rng.uniform(0.7, 1.3)
        elif grp == 'pothole a':
            for k in ['pothole_low', 'pothole_medium', 'pothole_high']:
                cfg_p['deduct_curves'][k]['a'] *= rng.uniform(0.7, 1.3)
        elif grp == 'pothole b':
            for k in ['pothole_low', 'pothole_medium', 'pothole_high']:
                cfg_p['deduct_curves'][k]['b'] *= rng.uniform(0.7, 1.3)
        elif grp == 'severity thresholds':
            # Perturb crack thresholds
            t_low = cfg_p['crack_severity']['low_threshold_px'] * rng.uniform(0.7, 1.3)
            t_high = cfg_p['crack_severity']['high_threshold_px'] * rng.uniform(0.7, 1.3)
            if t_low > t_high:
                t_low, t_high = t_high, t_low
            cfg_p['crack_severity']['low_threshold_px'] = t_low
            cfg_p['crack_severity']['high_threshold_px'] = t_high
            # Perturb pothole thresholds
            p_low = cfg_p['pothole_severity']['low_threshold_norm'] * rng.uniform(0.7, 1.3)
            p_high = cfg_p['pothole_severity']['high_threshold_norm'] * rng.uniform(0.7, 1.3)
            if p_low > p_high:
                p_low, p_high = p_high, p_low
            cfg_p['pothole_severity']['low_threshold_norm'] = p_low
            cfg_p['pothole_severity']['high_threshold_norm'] = p_high
        elif grp == 'beta':
            cfg_p['network_factor']['beta'] *= rng.uniform(0.7, 1.3)
        elif grp == 'network n_ref':
            cfg_p['network_factor']['n_ref'] *= rng.uniform(0.7, 1.3)
            
        draw_spci = np.array([compute_spci(pc, pp, config=cfg_p)['spci'] for pc, pp in val_preds])
        rho = float(spearmanr(baseline_spci_vals, draw_spci).statistic)
        rhos.append(rho)
        
    rhos = np.array(rhos)
    res_entry = {
        'min': float(np.min(rhos)),
        'p5': float(np.percentile(rhos, 5)),
        'median': float(np.median(rhos))
    }
    group_results[grp] = res_entry

print(f"{'Parameter Group':<24} | {'Min Rho':<12} | {'P5 Rho':<12} | {'Median Rho':<12}")
print("-" * 66)
for grp, res in group_results.items():
    print(f"{grp:<24} | {res['min']:12.4f} | {res['p5']:12.4f} | {res['median']:12.4f}")
print("=" * 66)

# Most sensitive group (lowest min rho or median rho)
most_sensitive_min = min(group_results.keys(), key=lambda k: group_results[k]['min'])
most_sensitive_median = min(group_results.keys(), key=lambda k: group_results[k]['median'])
print(f"\nMost Sensitive Group by Min Rho:    {most_sensitive_min} (min rho = {group_results[most_sensitive_min]['min']:.4f})")
print(f"Most Sensitive Group by Median Rho: {most_sensitive_median} (median rho = {group_results[most_sensitive_median]['median']:.4f})")

In [ ]:
# ==============================================================================
# 8. sPCI DISTRIBUTIONS ON VAL PER SOURCE & CONDITION BANDS
# ==============================================================================
print("\n" + "=" * 90)
print("sPCI DISTRIBUTIONS ON VALIDATION SPLIT (GT-BASED AND PREDICTED)")
print("=" * 90)

condition_bands_order = ["Good", "Satisfactory", "Fair", "Poor", "Very Poor", "Serious", "Failed"]

distribution_results = {}

for src in ['crack500', 'road_crack', 'pothole']:
    src_recs = [r for r in paired_val_records if r['source'] == src]
    n_total = len(src_recs)
    
    gt_spci = np.array([r['spci_gt'] for r in src_recs])
    pred_spci = np.array([r['spci_pred'] for r in src_recs])
    
    gt_classes = [get_condition_class(s) for s in gt_spci]
    pred_classes = [get_condition_class(s) for s in pred_spci]
    
    gt_band_counts = {b: gt_classes.count(b) for b in condition_bands_order}
    pred_band_counts = {b: pred_classes.count(b) for b in condition_bands_order}
    
    gt_ge99 = int(np.sum(gt_spci >= 99.0))
    pred_ge99 = int(np.sum(pred_spci >= 99.0))
    
    stats_src = {
        'n': n_total,
        'gt': {
            'min': float(np.min(gt_spci)),
            'p10': float(np.percentile(gt_spci, 10)),
            'median': float(np.median(gt_spci)),
            'p90': float(np.percentile(gt_spci, 90)),
            'max': float(np.max(gt_spci)),
            'band_counts': gt_band_counts,
            'count_ge_99': gt_ge99
        },
        'pred': {
            'min': float(np.min(pred_spci)),
            'p10': float(np.percentile(pred_spci, 10)),
            'median': float(np.median(pred_spci)),
            'p90': float(np.percentile(pred_spci, 90)),
            'max': float(np.max(pred_spci)),
            'band_counts': pred_band_counts,
            'count_ge_99': pred_ge99
        }
    }
    distribution_results[src] = stats_src
    
    print(f"\n--- SOURCE: {src.upper()} (N = {n_total}) ---")
    print(f"{'Type':<10} | {'Min':<8} | {'P10':<8} | {'Median':<8} | {'P90':<8} | {'Max':<8} | {'sPCI >= 99':<10}")
    print("-" * 66)
    print(f"{'GT':<10} | {stats_src['gt']['min']:8.2f} | {stats_src['gt']['p10']:8.2f} | {stats_src['gt']['median']:8.2f} | {stats_src['gt']['p90']:8.2f} | {stats_src['gt']['max']:8.2f} | {gt_ge99:10d}")
    print(f"{'Predicted':<10} | {stats_src['pred']['min']:8.2f} | {stats_src['pred']['p10']:8.2f} | {stats_src['pred']['median']:8.2f} | {stats_src['pred']['p90']:8.2f} | {stats_src['pred']['max']:8.2f} | {pred_ge99:10d}")
    print("\nCondition Band Counts (ASTM D6433 Figure 1 Non-Overlapping Bands):")
    for b in condition_bands_order:
        print(f"  • {b:<14}: GT = {gt_band_counts[b]:3d} | Predicted = {pred_band_counts[b]:3d}")

# Road crack specific subset analysis
rc_recs = [r for r in paired_val_records if r['source'] == 'road_crack']
rc_with_gt_crack = [r for r in rc_recs if (r['gt_t_crack'] is not None and np.any(r['gt_t_crack'] > 0))]
rc_empty_gt_crack = [r for r in rc_recs if (r['gt_t_crack'] is None or not np.any(r['gt_t_crack'] > 0))]

print("\n" + "=" * 90)
print("ROAD_CRACK SUBSET ANALYSIS (CRACK-BEARING VS EMPTY GT MASKS)")
print("=" * 90)
print(f"Total road_crack images: {len(rc_recs)}")
print(f"Images with >= 1 GT crack pixel: {len(rc_with_gt_crack)}")
print(f"Images with empty GT mask:       {len(rc_empty_gt_crack)}")

# For images with >= 1 GT crack pixel: Spearman rho & MAE
gt_spci_bearing = np.array([r['spci_gt'] for r in rc_with_gt_crack])
pred_spci_bearing = np.array([r['spci_pred'] for r in rc_with_gt_crack])
rho_bearing = float(spearmanr(gt_spci_bearing, pred_spci_bearing).statistic)
mae_bearing = float(np.mean(np.abs(gt_spci_bearing - pred_spci_bearing)))

print(f"\n1. For the {len(rc_with_gt_crack)} images with >= 1 GT crack pixel:")
print(f"   • Spearman rho: {rho_bearing:.4f}")
print(f"   • MAE:          {mae_bearing:.2f} points")

# For empty-GT images: count with predicted sPCI < 95 and distribution
pred_spci_empty = np.array([r['spci_pred'] for r in rc_empty_gt_crack])
n_lt95 = int(np.sum(pred_spci_empty < 95.0))
dist_empty = {
    'count': len(rc_empty_gt_crack),
    'count_pred_lt_95': n_lt95,
    'frac_pred_lt_95': float(n_lt95 / len(rc_empty_gt_crack)),
    'min': float(np.min(pred_spci_empty)),
    'p10': float(np.percentile(pred_spci_empty, 10)),
    'median': float(np.median(pred_spci_empty)),
    'p90': float(np.percentile(pred_spci_empty, 90)),
    'max': float(np.max(pred_spci_empty))
}

print(f"\n2. For the {len(rc_empty_gt_crack)} empty-GT images:")
print(f"   • Count with predicted sPCI < 95: {n_lt95} / {len(rc_empty_gt_crack)} ({dist_empty['frac_pred_lt_95']*100:.1f}%)")
print(f"   • Distribution of predicted sPCI:")
print(f"     Min:    {dist_empty['min']:.2f}")
print(f"     P10:    {dist_empty['p10']:.2f}")
print(f"     Median: {dist_empty['median']:.2f}")
print(f"     P90:    {dist_empty['p90']:.2f}")
print(f"     Max:    {dist_empty['max']:.2f}")
print("=" * 90)

road_crack_subsets = {
    'bearing_n': len(rc_with_gt_crack),
    'bearing_rho': rho_bearing,
    'bearing_mae': mae_bearing,
    'empty_stats': dist_empty
}

In [ ]:
# ==============================================================================
# 9. OPTIONAL NOISE FLOOR (TAU) SWEEP ON VALIDATION SPLIT
# ==============================================================================
print("\n" + "=" * 90)
print("NOISE FLOOR (TAU) TUNING CURVE ON VAL (SHORT SIDES: LENGTH_PX / 512)")
print("=" * 90)

tau_values = [0.0, 0.1, 0.2, 0.3, 0.5]
tau_tuning_records = []

for tau in tau_values:
    # Compute sPCI on VAL with noise floor tau
    c500_gt, c500_pred = [], []
    rc_gt, rc_pred = [], []
    
    for r in paired_val_records:
        src = r['source']
        if src in ['crack500', 'road_crack']:
            s_pred = compute_spci(r['pred_p_crack'], r['pred_p_pot'], config=spci_cfg, noise_floor_tau=tau)['spci']
            s_gt = r['spci_gt']
            if src == 'crack500':
                c500_gt.append(s_gt)
                c500_pred.append(s_pred)
            elif src == 'road_crack':
                rc_gt.append(s_gt)
                rc_pred.append(s_pred)
                
    rho_c500 = float(spearmanr(c500_gt, c500_pred).statistic)
    mae_c500 = float(np.mean(np.abs(np.array(c500_gt) - np.array(c500_pred))))
    
    rho_rc = float(spearmanr(rc_gt, rc_pred).statistic)
    mae_rc = float(np.mean(np.abs(np.array(rc_gt) - np.array(rc_pred))))
    
    tau_tuning_records.append({
        'tau': tau,
        'crack500_rho': rho_c500,
        'crack500_mae': mae_c500,
        'road_crack_rho': rho_rc,
        'road_crack_mae': mae_rc
    })

print(f"{'Tau (Short Sides)':<18} | {'CRACK500 Rho':<14} | {'CRACK500 MAE':<14} | {'Road Crack Rho':<16} | {'Road Crack MAE':<16}")
print("-" * 86)
for row in tau_tuning_records:
    print(f"{row['tau']:<18.2f} | {row['crack500_rho']:14.4f} | {row['crack500_mae']:14.2f} | {row['road_crack_rho']:16.4f} | {row['road_crack_mae']:16.2f}")
print("=" * 86)

# Evaluate decision rule:
# "Adopt tau > 0 only if road_crack rho improves by >= 0.03 and CRACK500 rho drops by no more than 0.01; otherwise keep tau = 0."
baseline_rc_rho = tau_tuning_records[0]['road_crack_rho']
baseline_c500_rho = tau_tuning_records[0]['crack500_rho']

adopted_tau = 0.0
adoption_reason = "No tau > 0 met both criteria: road_crack rho delta >= +0.03 and crack500 rho delta >= -0.01."

for row in tau_tuning_records[1:]:
    t = row['tau']
    delta_rc = row['road_crack_rho'] - baseline_rc_rho
    delta_c500 = row['crack500_rho'] - baseline_c500_rho
    print(f"\nEvaluation for tau = {t}:")
    print(f"  • Road crack rho change: {delta_rc:+.4f} (Required: >= +0.03)")
    print(f"  • CRACK500 rho change:   {delta_c500:+.4f} (Required: >= -0.01)")
    if delta_rc >= 0.03 and delta_c500 >= -0.01:
        adopted_tau = t
        adoption_reason = f"tau = {t} adopted: road_crack rho improved by {delta_rc:+.4f} >= 0.03 and CRACK500 rho changed by {delta_c500:+.4f} >= -0.01."
        print(f"  --> Meets criteria!")
    else:
        print(f"  --> Does NOT meet criteria.")

print(f"\nFINAL RULE DECISION: Adopted tau = {adopted_tau}")
print(f"Rationale: {adoption_reason}")

In [ ]:
# ==============================================================================
# 10. EXPORT STRUCTURED SUMMARY JSON
# ==============================================================================
summary_payload = {
    'weights_sha256': sha256,
    'trainval_records': len(gt_features_df),
    'calibration': {
        'p33_width_px': p33_width,
        'median_width_px': median_width,
        'p80_width_px': p80_width,
        'n_ref': n_ref_val
    },
    'feat_spearman_rho': feat_corr_results,
    'spci_val_comparison': spci_comparison_results,
    'sensitivity_7_groups': group_results,
    'most_sensitive_group': {
        'by_min': most_sensitive_min,
        'by_median': most_sensitive_median
    },
    'val_distributions': distribution_results,
    'road_crack_subsets': road_crack_subsets,
    'tau_sweep': tau_tuning_records,
    'adopted_tau': adopted_tau,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}

with open("/kaggle/working/step_4b_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)
print("[OK] Exported /kaggle/working/step_4b_summary.json successfully.")